In [1]:
import re, unicodedata
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error


In [2]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

def run_baseline(df, label):
    X = pd.get_dummies(
        df[["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]],
        columns=["position"], dtype=float,
    )
    train = df["signing_year"] <= 2023
    test = df["signing_year"] >= 2024

    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[train], df.loc[train, "aav_m"])

    pred = model.predict(X[test])
    actual = df.loc[test, "aav_m"]
    naive = np.full(len(actual), df.loc[train, "aav_m"].median())

    print(f"{label} | train {train.sum()} | test {test.sum()}")
    print(f"  Model MAE: ${mean_absolute_error(actual, pred):.2f}M")
    print(f"  Naive MAE (always guess the median): ${mean_absolute_error(actual, naive):.2f}M")

run_baseline(d, "All contracts")
run_baseline(d[d["total_value"] >= 5_000_000], "$5M+ contracts only")

All contracts | train 1103 | test 448
  Model MAE: $3.53M
  Naive MAE (always guess the median): $5.62M
$5M+ contracts only | train 561 | test 213
  Model MAE: $4.38M
  Naive MAE (always guess the median): $6.57M


In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

X = pd.get_dummies(
    d[["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]],
    columns=["position"], dtype=float,
)
train = d["signing_year"] <= 2023
test = d["signing_year"] >= 2024

model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
model.fit(X[train], d.loc[train, "aav_m"])

d["pred_m"] = model.predict(X)
d["error_m"] = d["pred_m"] - d["aav_m"]   # negative = model guessed too low

print("Average error by year (in $M, negative = guessed too low):")
print(d.groupby("signing_year")["error_m"].mean().round(2))

t = d[test].copy()
t["abs_err"] = t["error_m"].abs()
print("\nBiggest misses in the test years:")
print(t.nlargest(10, "abs_err")[
    ["player_name", "signing_year", "position", "war_w", "age_at_signing", "aav_m", "pred_m"]
].round(1))

Average error by year (in $M, negative = guessed too low):
signing_year
2015    0.59
2016    0.94
2017   -0.23
2018    0.67
2019    0.58
2020    0.84
2021    0.69
2022   -1.49
2023   -2.05
2024   -0.84
2025   -1.13
2026   -1.86
Name: error_m, dtype: float64

Biggest misses in the test years:
         player_name  signing_year position  war_w  age_at_signing  aav_m  \
626      Kyle Tucker          2026       RF    4.8            28.9   60.0   
631      Bo Bichette          2026       SS    2.5            27.7   42.0   
627     Alex Bregman          2026       3B    4.1            31.7   35.0   
1417     Blake Snell          2025       SP    3.6            32.0   36.4   
629      Pete Alonso          2026       1B    3.0            31.0   31.0   
557        Juan Soto          2025       RF    6.5            26.1   51.0   
1434  Walker Buehler          2025       SP   -0.4            30.3   21.0   
628   Cody Bellinger          2026       CF    4.0            30.4   32.5   
559     Alex B

In [4]:
from sklearn.metrics import mean_absolute_error

def fit_eval(df, use_year=False, use_log=False):
    cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]
    X = pd.get_dummies(df[cols], columns=["position"], dtype=float)
    if use_year:
        X["year"] = df["signing_year"] - 2015
    tr = df["signing_year"] <= 2023
    te = df["signing_year"] >= 2024
    y = np.log(df["aav_m"]) if use_log else df["aav_m"]
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[tr], y[tr])
    pred = model.predict(X[te])
    if use_log:
        pred = np.exp(pred)
    actual = df.loc[te, "aav_m"].values
    return mean_absolute_error(actual, pred), (pred - actual).mean()

variants = [
    ("Baseline", False, False),
    ("+ year trend", True, False),
    ("+ year trend, log target", True, True),
]

for label, subset in [("All contracts", d), ("$5M+ only", d[d["total_value"] >= 5_000_000])]:
    print(label)
    for name, uy, ul in variants:
        mae, bias = fit_eval(subset, uy, ul)
        print(f"  {name:28s} MAE ${mae:.2f}M | avg error ${bias:+.2f}M")

All contracts
  Baseline                     MAE $3.53M | avg error $-1.28M
  + year trend                 MAE $3.74M | avg error $+0.34M
  + year trend, log target     MAE $3.91M | avg error $-1.03M
$5M+ only
  Baseline                     MAE $4.38M | avg error $-2.65M
  + year trend                 MAE $4.00M | avg error $-0.16M
  + year trend, log target     MAE $4.05M | avg error $-1.19M


In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

# Season-by-season WAR lookup from the files saved earlier
bh = pd.read_csv("../data/processed/bwar_hitters.csv")
bp = pd.read_csv("../data/processed/bwar_pitchers.csv")
war_lookup = {}
for t, s in [("hitter", bh), ("pitcher", bp)]:
    for mid, yr, w in zip(s["mlb_ID"], s["year_ID"], s["WAR"]):
        war_lookup[(t, mid, yr)] = w

# Best single season in the 5 years before signing
d["peak5"] = [
    max(war_lookup.get((t, i, y - k), 0.0) for k in range(5))
    for t, i, y in zip(d["player_type"], d["mlb_ID"], d["stat_year"])
]

def fit_eval(df, extra=()):
    cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"] + list(extra)
    X = pd.get_dummies(df[cols], columns=["position"], dtype=float)
    X["year"] = df["signing_year"] - 2015
    tr = df["signing_year"] <= 2023
    te = df["signing_year"] >= 2024
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[tr], df.loc[tr, "aav_m"])
    pred = model.predict(X[te])
    actual = df.loc[te, "aav_m"].values
    return mean_absolute_error(actual, pred), (pred - actual).mean()

for label, subset in [("All contracts", d), ("$5M+ only", d[d["total_value"] >= 5_000_000])]:
    print(label)
    for name, extra in [("Year trend", ()), ("Year trend + peak WAR", ("peak5",))]:
        mae, bias = fit_eval(subset, extra)
        print(f"  {name:24s} MAE ${mae:.2f}M | avg error ${bias:+.2f}M")

print(d[d["player_name"] == "Walker Buehler"][["signing_year", "war_w", "peak5", "aav_m"]])

All contracts
  Year trend               MAE $3.74M | avg error $+0.34M
  Year trend + peak WAR    MAE $3.71M | avg error $+0.19M
$5M+ only
  Year trend               MAE $4.00M | avg error $-0.16M
  Year trend + peak WAR    MAE $3.99M | avg error $-0.59M
      signing_year  war_w  peak5  aav_m
1434          2025  -0.39   7.13  21.05


In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d = d[d["total_value"] >= 5_000_000].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015

sav = {
    "hitter": pd.read_csv("../data/processed/savant_hitters.csv"),
    "pitcher": pd.read_csv("../data/processed/savant_pitchers.csv"),
}
feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}

def savant_avg(df, t):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid, sy in zip(df["mlb_ID"], df["stat_year"]):
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), int(sy) - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

def compare(t):
    df = d[d["player_type"] == t].copy()
    f = feats[t]
    total = len(df)
    df = pd.concat([df, savant_avg(df, t)], axis=1)
    df = df[df[f[0]].notna()].copy()      # require the main quality metric
    df[f] = df[f].fillna(50)              # other gaps: league-median percentile
    tr = (df["signing_year"] <= 2023).values
    te = ~tr
    print(f"\n{t.upper()}S: {len(df)} of {total} contracts have Savant data "
          f"| train {tr.sum()} | test {te.sum()}")

    def fit(cols):
        X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
        model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        model.fit(X[tr], df.loc[tr, "aav_m"])
        return mean_absolute_error(df.loc[te, "aav_m"], model.predict(X[te]))

    base = ["age_at_signing", "age_sq", "qualifying_offer", "year"]
    naive = mean_absolute_error(df.loc[te, "aav_m"],
                                np.full(te.sum(), df.loc[tr, "aav_m"].median()))
    print(f"  Naive (median)              MAE ${naive:.2f}M")
    print(f"  A: WAR model                MAE ${fit(base + ['war_w']):.2f}M")
    print(f"  B: WAR + Savant             MAE ${fit(base + ['war_w'] + f):.2f}M")
    print(f"  C: Savant only (no WAR)     MAE ${fit(base + f):.2f}M")

compare("hitter")
compare("pitcher")


HITTERS: 291 of 328 contracts have Savant data | train 214 | test 77
  Naive (median)              MAE $7.94M
  A: WAR model                MAE $4.93M
  B: WAR + Savant             MAE $4.18M
  C: Savant only (no WAR)     MAE $4.94M

PITCHERS: 398 of 446 contracts have Savant data | train 271 | test 127
  Naive (median)              MAE $5.96M
  A: WAR model                MAE $3.69M
  B: WAR + Savant             MAE $3.43M
  C: Savant only (no WAR)     MAE $3.87M


In [7]:
def rolling(t, test_years=range(2021, 2027)):
    df = d[d["player_type"] == t].copy()
    f = feats[t]
    df = pd.concat([df, savant_avg(df, t)], axis=1)
    df = df[df[f[0]].notna()].copy()
    df[f] = df[f].fillna(50)
    base = ["age_at_signing", "age_sq", "qualifying_offer", "year"]

    def fit_predict(cols, tr, te):
        X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
        model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        model.fit(X[tr], df.loc[tr, "aav_m"])
        return model.predict(X[te])

    rows = []
    for yr in test_years:
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        if te.sum() == 0 or tr.sum() < 30:
            continue
        actual = df.loc[te, "aav_m"].values
        a = fit_predict(base + ["war_w"], tr, te)
        b = fit_predict(base + ["war_w"] + f, tr, te)
        rows.append((yr, te.sum(), np.abs(a - actual).mean(), np.abs(b - actual).mean()))

    out = pd.DataFrame(rows, columns=["test_year", "n", "MAE_A_WAR", "MAE_B_WAR_Savant"])
    print(f"\n{t.upper()}S")
    print(out.round(2).to_string(index=False))
    pooled_a = (out["n"] * out["MAE_A_WAR"]).sum() / out["n"].sum()
    pooled_b = (out["n"] * out["MAE_B_WAR_Savant"]).sum() / out["n"].sum()
    print(f"Pooled MAE: A ${pooled_a:.2f}M | B ${pooled_b:.2f}M")
    print("B beats A in", (out["MAE_B_WAR_Savant"] < out["MAE_A_WAR"]).sum(), "of", len(out), "years")

rolling("hitter")
rolling("pitcher")


HITTERS
 test_year  n  MAE_A_WAR  MAE_B_WAR_Savant
      2021 27       2.98              3.08
      2022 31       3.70              3.77
      2023 38       3.12              3.19
      2024 21       3.96              3.31
      2025 29       4.14              3.35
      2026 27       6.07              5.45
Pooled MAE: A $3.93M | B $3.67M
B beats A in 3 of 6 years

PITCHERS
 test_year  n  MAE_A_WAR  MAE_B_WAR_Savant
      2021 33       2.98              2.95
      2022 48       3.20              3.05
      2023 47       3.55              3.47
      2024 41       3.41              3.32
      2025 44       4.11              3.90
      2026 42       3.38              2.92
Pooled MAE: A $3.46M | B $3.28M
B beats A in 6 of 6 years


In [8]:
old = pd.read_csv("../outputs/fa_2027_predictions_v1.csv")
new = pd.read_csv("../outputs/fa_2027_predictions.csv")
cmp = old[["player_name", "pred_aav_m"]].merge(
    new[["player_name", "pred_aav_m"]], on="player_name", suffixes=("_old", "_new"))
cmp["change"] = (cmp["pred_aav_m_new"] - cmp["pred_aav_m_old"]).round(1)
print("Average change: %+.1fM" % cmp["change"].mean())
print(cmp.sort_values("pred_aav_m_old", ascending=False).head(15).to_string(index=False))

Average change: -0.1M
     player_name  pred_aav_m_old  pred_aav_m_new  change
    Tarik Skubal            31.8            32.2     0.4
    Seiya Suzuki            23.8            24.0     0.2
  Drew Rasmussen            22.6            22.9     0.3
 Randy Arozarena            21.3            21.3     0.0
Ronald Acuña Jr.            20.8            20.8     0.0
        Ian Happ            19.2            19.2     0.0
  Freddy Peralta            19.2            19.3     0.1
   Trent Grisham            18.9            19.0     0.1
    Michael King            18.9            19.1     0.2
   Trevor Rogers            18.2            18.2     0.0
    Ryan Jeffers            18.0            18.1     0.1
    Nick Pivetta            17.8            18.0     0.2
   J.P. Crawford            17.6            17.7     0.1
   Nick Martinez            17.3            17.4     0.1
      Sonny Gray            16.8            16.9     0.1


In [9]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = pd.read_csv("../data/processed/contracts_model.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")]
d = d[(d["total_value"] >= 5_000_000) & (d["player_type"] == "hitter")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015

cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "year", "position"]
X = pd.get_dummies(d[cols], columns=["position"], dtype=float)

rows = []
for yr in range(2021, 2027):
    tr = (d["signing_year"] < yr).values
    te = (d["signing_year"] == yr).values
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    m.fit(X[tr], d.loc[tr, "aav_m"])
    t = d[te].copy()
    t["pred"] = m.predict(X[te])
    rows.append(t)

r = pd.concat(rows)
r["err"] = r["pred"] - r["aav_m"]          # positive = model guessed too high
r["war_band"] = pd.cut(r["war_w"], [-10, 0.5, 1.3, 2.3, 3.5, 20],
                       labels=["under 0.5", "0.5-1.3", "1.3-2.3", "2.3-3.5", "3.5+"])

for label, sub in [("All test years (2021-2026)", r),
                   ("2025-2026 only", r[r["signing_year"] >= 2025])]:
    print(label)
    print(sub.groupby("war_band", observed=True).agg(
        n=("err", "size"), actual=("aav_m", "mean"),
        predicted=("pred", "mean"), bias=("err", "mean")).round(1))
    print()

All test years (2021-2026)
            n  actual  predicted  bias
war_band                              
under 0.5   8     6.8        5.7  -1.2
0.5-1.3    53     7.2        8.2   1.0
1.3-2.3    56    11.0       10.9  -0.1
2.3-3.5    38    17.5       15.3  -2.1
3.5+       23    28.1       21.7  -6.4

2025-2026 only
            n  actual  predicted  bias
war_band                              
under 0.5   2     6.4        7.6   1.2
0.5-1.3    15     7.0        9.1   2.0
1.3-2.3    17    10.6       12.0   1.4
2.3-3.5    16    19.9       17.8  -2.2
3.5+        8    33.2       23.3  -9.9



In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d0 = pd.read_csv("../data/processed/contracts_model.csv")
d0 = d0[d0["age_at_signing"].notna() & (d0["player_name"] != "Shohei Ohtani")]
d0 = d0[d0["total_value"] >= 5_000_000].copy()
d0["aav_m"] = d0["aav"] / 1e6
d0["age_sq"] = (d0["age_at_signing"] - 31) ** 2
d0["year"] = d0["signing_year"] - 2015
d0["war_sq"] = d0["war_w"].clip(lower=0) ** 2

def rolling(df, cols):
    X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
    rows = []
    for yr in range(2021, 2027):
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        m.fit(X[tr], df.loc[tr, "aav_m"])
        t = df[te].copy()
        t["pred"] = m.predict(X[te])
        rows.append(t)
    r = pd.concat(rows)
    r["err"] = r["pred"] - r["aav_m"]
    return r

base = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "year"]
for t in ["hitter", "pitcher"]:
    df = d0[d0["player_type"] == t]
    print(t.upper() + "S")
    for name, cols in [("Linear WAR", base), ("+ WAR squared", base + ["war_sq"])]:
        r = rolling(df, cols)
        hi = r[r["war_w"] >= 3.5]
        mid = r[(r["war_w"] >= 2.3) & (r["war_w"] < 3.5)]
        print(f"  {name:14s} MAE ${r['err'].abs().mean():.2f}M"
              f" | bias 3.5+ WAR: ${hi['err'].mean():+.1f}M (n={len(hi)})"
              f" | bias 2.3-3.5: ${mid['err'].mean():+.1f}M (n={len(mid)})")

HITTERS
  Linear WAR     MAE $3.83M | bias 3.5+ WAR: $-6.4M (n=23) | bias 2.3-3.5: $-2.1M (n=38)
  + WAR squared  MAE $3.88M | bias 3.5+ WAR: $-6.0M (n=23) | bias 2.3-3.5: $-2.8M (n=38)
PITCHERS
  Linear WAR     MAE $3.47M | bias 3.5+ WAR: $-5.2M (n=10) | bias 2.3-3.5: $-1.9M (n=25)
  + WAR squared  MAE $3.44M | bias 3.5+ WAR: $-5.0M (n=10) | bias 2.3-3.5: $-2.6M (n=25)
